# KL 散度与训练稳定性

RL 训练有一个常见问题：**策略崩塌**。模型为了追求高奖励，可能逐渐丢失原有的语言能力，最终输出退化为固定模式。

本节介绍两个关键的稳定性机制：KL 散度正则化和 Entropy Bonus。

---

## 1. 策略崩塌现象

策略崩塌是指模型的输出分布变得过于集中（熵趋近于 0），丧失了探索能力。在 Wordle 训练中的典型表现：

```text
策略崩塌的典型表现

正常训练:
  step 1:   entropy=0.56, correct=15%, response_length=1700
  step 75:  entropy=0.28, correct=40%, response_length=1550
  -> 稳定，entropy 缓慢下降

策略崩塌:
  step 1:   entropy=0.56, correct=15%, response_length=1700
  step 75:  entropy=0.28, correct=40%, response_length=1550
  step 100: entropy=0.001, correct=0%,  response_length=3232 (全部打满)
  -> entropy 突然崩塌，输出退化为固定模式
```

崩塌后，模型每次输出几乎完全相同的文本（entropy 约等于 0），不再尝试新策略，correct 率归零，response 长度全部打满。

---

## 2. KL 散度正则化

**KL 散度**（Kullback-Leibler Divergence）衡量两个策略分布的差异。在 RL 训练中，我们用它来限制当前策略 π 与参考策略 π_ref 之间的距离。

### 2.1 两种 KL 机制

verl 提供两种独立的 KL 机制：

| 机制 | 配置 | 作用位置 |
|------|------|----------|
| **KL in reward** | `algorithm.use_kl_in_reward` + `kl_ctrl.kl_coef` | 在 reward 中减去惩罚项 |
| **KL loss** | `actor.use_kl_loss` + `actor.kl_loss_coef` | 在 loss 中加入惩罚项 |

我们的训练使用 **KL loss** 方式：

```text
KL loss 方式

total_loss = pg_loss + kl_loss_coef * KL(pi || pi_ref)

当前配置:
  actor.use_kl_loss = True
  actor.kl_loss_coef = 0.001
  actor.kl_loss_type = low_var_kl  (低方差 KL，数值更稳定)

效果: 模型偏离参考策略越远，KL loss 越大，总 loss 越高
-> 梯度会拉回模型，防止偏离太远
```

### 2.2 low_var_kl

verl 支持多种 KL 计算方式，我们使用 `low_var_kl`（低方差 KL）。相比标准 KL，它在数值上更稳定，特别适合 token 级别的策略分布比较。

### 2.3 KL coef 的选择

KL coef 控制正则化强度，是一个需要平衡的超参数：

| KL coef | 效果 | 风险 |
|---------|------|------|
| 0.0001 | 几乎无约束 | 策略漂移过大，可能崩塌 |
| 0.001 | 轻微约束 | 多数环境够用，漂移较大时偏弱 |
| 0.01 | 中等约束 | 配合 entropy_coeff 使用，推荐 |
| 0.1 | 强约束 | 模型无法学习新策略 |

---

## 3. Entropy Bonus

**Entropy Bonus** 是另一种稳定性机制：在 loss 中加入一个鼓励高熵（高探索性）的项。

```text
Entropy Bonus

total_loss = pg_loss + kl_loss_coef * KL_loss - entropy_coeff * H(pi)

H(pi) = 策略的熵 (entropy)
entropy_coeff = 熵系数

效果: 熵越低 (探索性越差)，-entropy_coeff * H(pi) 越大，loss 越高
-> 梯度会推高熵，鼓励模型保持探索能力
```

### 3.1 Entropy 的平衡

Entropy bonus 同样需要平衡：

| entropy_coeff | 效果 | 风险 |
|---------------|------|------|
| 0 (默认) | 无约束 | 几乎必崩塌 |
| 0.001 | 轻微探索 | 多数环境稳定，但熵偏低（~0.1）|
| 0.002 | 适度探索 | 熵稳定（~0.2），多数环境最优 |
| 0.005 | 较强探索 | ⚠️ 边界值，部分环境熵飙升 |
| 0.01+ | 过强探索 | 几乎必飙升（0.56→4.51 in 25 步）|

> ⚠️ **entropy_coeff 的最优值受 SFT 模型状态、随机种子、硬件浮点差异影响**，0.001~0.005 之间的边界会随环境漂移。建议从 0.001 起步，跑 30 步观察 entropy 趋势再决定是否上调。

### 3.2 三种力的平衡

RL 训练的稳定性是三种力的动态平衡：

```text
RL 训练中的三种力

pg_loss：优化 reward，让模型更倾向于高奖励输出，但也会让分布变得更确定，entropy 下降。
kl_loss：把当前策略拉回参考策略附近，限制策略漂移。
entropy_bonus：鼓励探索，避免输出过早收缩到固定模式。
```


三者不是越大越好，而是要保持合适的平衡：pg_loss 过强容易策略崩塌，entropy_bonus 过强又可能探索失控，kl_loss 则负责限制模型偏离参考策略太远。

<img src="./images/three_forces.png" alt="RL 训练三种力平衡" width="90%">

图中的平衡点表示稳定训练状态：模型既能朝高奖励方向学习，又不会过度偏离参考策略，也不会失去必要的探索能力。

---

## 4. 训练健康度检查

在训练过程中，可以通过以下指标判断训练是否健康：

| 指标 | 健康范围 | 危险信号 |
|------|----------|----------|
| entropy | 缓慢下降 | 突然跌到 0（崩塌）或持续飙升（失控） |
| kl_loss | 缓慢上升 | 飙升过快（漂移过大） |
| grad_norm | 稳定 | 突然飙升（梯度爆炸） |
| response_length | 稳定 | 全部打满（输出退化） |
| reward | 上升趋势 | 突然归零（策略崩溃） |

> 这些指标会在第 3 章训练实战中实际观察和解读。

---

## 课后练习

1. （判断题）策略崩塌是指模型的 entropy 趋近于 0，丧失探索能力，输出退化为固定模式。

2. （判断题）KL 散度正则化的作用是限制当前策略与参考策略之间的距离，防止策略漂移过大。

3. （判断题）Entropy bonus 的作用是降低模型的探索性，让模型更专注于高奖励的输出。

4. （单选题）KL loss 中 `kl_loss_coef=0.001` 的作用是？
    A. 完全禁止策略变化
    B. 轻微约束策略漂移
    C. 加速训练
    D. 计算奖励

5. （单选题）以下哪个指标是策略崩塌的典型信号？
    A. entropy 缓慢下降
    B. reward 持续上升
    C. entropy 突然跌到 0，response_length 全部打满
    D. kl_loss 缓慢上升

6. （多选题）RL 训练中的三种力包括哪些？
    A. 策略梯度 (pg_loss)
    B. KL 正则 (kl_loss)
    C. 熵奖励 (entropy_bonus)
    D. Critic 价值估计

7. （多选题）以下哪些是 entropy_coeff 设置过大可能导致的问题？
    A. Entropy 失控飙升
    B. 模型过度探索，不收敛
    C. Reward 提升靠 partial 而非 correct
    D. 策略崩塌

In [ ]:
!cat ../cann-learning-hub/tutorials/rl_training_pipeline/02_rl_core_concepts/answer/02.04_answer.txt